In [1]:
# ==============================================================================
# PIPELINE 3 (P3) — TOKEN RELATIONAL DISTILLATION + EMA
# P1 + RKD-style patch similarity alignment + EMA model for validation:
#   Pairwise cosine similarity matrices (49x49) aligned between
#   ResNet layer4 spatial locations and spatially-pooled DeiT patch tokens
# All 4 scales run automatically: 10% → 25% → 50% → 100%
# ==============================================================================

import os, random, math
import numpy as np
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
import torchvision, torchvision.transforms as transforms
import timm

SEED            = 67
BATCH_SIZE      = 64
EPOCHS          = 10
LEARNING_RATE   = 8e-4
LABEL_SMOOTHING = 0.1
DROP_PATH_RATE  = 0.1
KD_TEMPERATURE  = 4.5
DECAY_RATE      = 0.10
TASK_WEIGHT     = 0.3
DISTILL_WEIGHT  = 0.7
TOKEN_REL_WEIGHT= 0.3   # initial weight for token relation loss, also annealed
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_DIR       = '/kaggle/input/datasets/melikechan/cifar100/cifar100/train'
TEST_DIR        = '/kaggle/input/datasets/melikechan/cifar100/cifar100/test'
MODEL_PATH      = '/kaggle/input/models/totallyapoorv/resnetoncifar100/pytorch/default/1/resnetall.pth'
DATA_SCALES     = ['10%', '25%', '50%', '100%']
SCALE_MAP       = {'10%': 0.10, '25%': 0.25, '50%': 0.50, '100%': 1.0}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED); torch.backends.cudnn.deterministic = True

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])
transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

full_trainset = torchvision.datasets.ImageFolder(TRAIN_DIR, transform=transform_train)
testset       = torchvision.datasets.ImageFolder(TEST_DIR,  transform=transform_test)
all_indices   = list(range(len(full_trainset)))
random.shuffle(all_indices)

criterion_kl = nn.KLDivLoss(reduction='batchmean')

def soft_kd_loss(s_logits, t_logits, T=KD_TEMPERATURE):
    return criterion_kl(F.log_softmax(s_logits/T, dim=1),
                        F.softmax(t_logits/T, dim=1)) * (T**2)

def cosine_sem_loss(t_feat, s_feat, proj):
    return 1.0 - F.cosine_similarity(t_feat, proj(s_feat), dim=1).mean()

def token_relation_loss(t_spatial, s_patches):
    # t_spatial : [B, 512, 7, 7]
    # s_patches : [B, 196, 192]  patch tokens from final norm layer
    B = t_spatial.size(0)
    t = F.normalize(t_spatial.view(B, 512, -1).transpose(1, 2), dim=-1)  # [B,49,512]
    t_rel = torch.bmm(t, t.transpose(1, 2))                               # [B,49,49]
    s = s_patches.transpose(1, 2).view(B, 192, 14, 14)
    s = F.avg_pool2d(s, kernel_size=2, stride=2)                          # [B,192,7,7]
    s = F.normalize(s.view(B, 192, -1).transpose(1, 2), dim=-1)           # [B,49,192]
    s_rel = torch.bmm(s, s.transpose(1, 2))                               # [B,49,49]
    return F.mse_loss(s_rel, t_rel.detach())

def update_ema(model, ema_p, decay):
    with torch.no_grad():
        for k, v in model.state_dict().items():
            ema_p[k] = decay * ema_p[k] + (1 - decay) * v.float()

def validate_ema(model, ema_p, testloader):
    orig = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict({k: v.to(DEVICE) for k, v in ema_p.items()})
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            out = model(inputs)
            if isinstance(out, tuple): out = (out[0] + out[1]) / 2
            correct += out.max(1)[1].eq(targets).sum().item(); total += targets.size(0)
    model.load_state_dict(orig)
    return 100. * correct / total

def make_scheduler(opt, total_epochs, base_lr, min_lr=1e-6):
    def lr_fn(epoch):
        p = epoch / max(total_epochs - 1, 1)
        c = 0.5 * (1 + math.cos(math.pi * p))
        return (min_lr / base_lr) + (1 - min_lr / base_lr) * c
    return torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

results_p3 = {}

for DATA_SCALE in DATA_SCALES:
    print(f"\n{'='*65}\n  P3 | {DATA_SCALE}\n{'='*65}")
    CKPT = f"p3_{DATA_SCALE}.pth"

    n = int(len(full_trainset) * SCALE_MAP[DATA_SCALE])
    trainloader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(full_trainset, all_indices[:n]),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    testloader = torch.utils.data.DataLoader(
        testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    print(f"  {n} images | {len(trainloader)} batches/epoch")





    
    teacher = torchvision.models.resnet18(weights=None)
    teacher.fc = nn.Linear(teacher.fc.in_features, 100)
    teacher.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    teacher = teacher.to(DEVICE).eval()

    student = timm.create_model('deit_tiny_distilled_patch16_224',
                                 pretrained=False, num_classes=100,
                                 drop_path_rate=DROP_PATH_RATE)
    student.set_distilled_training(True)
    student = student.to(DEVICE)

    proj_head = nn.Linear(192, 512).to(DEVICE)
    criterion_ce = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    # Hooks: semantic features + spatial features + student patch tokens
    cache = {}
    teacher.avgpool.register_forward_hook(
        lambda m, i, o: cache.update({'t_pool': o.view(o.size(0), -1)}))
    teacher.layer4.register_forward_hook(
        lambda m, i, o: cache.update({'t_spatial': o}))          # [B, 512, 7, 7]
    student.head_dist.register_forward_hook(
        lambda m, i, o: cache.update({'s_dist': i[0]}))
    student.norm.register_forward_hook(
        lambda m, i, o: cache.update({'s_patches': o[:, 2:, :]})) # [B, 196, 192]

    # EMA initialised from fresh student weights

    # compute EMA_DECAY dynamically based on actual update count
    #

    updates_per_run = len(trainloader) * EPOCHS
    # Target: initialization contributes <5% by end of training
    # decay^updates = 0.05  →  decay = 0.05^(1/updates)
    EMA_DECAY = 0.030 ** (1.0 / updates_per_run)
    print(f"  EMA_DECAY set to {EMA_DECAY:.6f} for {updates_per_run} total updates")
    
    # Re-initialise ema_params after this line (before the checkpoint resume block)
    ema_params = {k: v.clone().float().detach() for k, v in student.state_dict().items()}


    params = list(student.parameters()) + list(proj_head.parameters())
    opt    = optim.AdamW(params, lr=LEARNING_RATE, weight_decay=0.05)
    sched  = make_scheduler(opt, EPOCHS, LEARNING_RATE)

    start_epoch = 0; best_acc = 0.0
    if os.path.exists(CKPT):
        ck = torch.load(CKPT)
        student.load_state_dict(ck['model']); proj_head.load_state_dict(ck['proj'])
        opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
        ema_params = ck['ema']
        start_epoch = ck['epoch'] + 1; best_acc = ck['best_acc']
        print(f"  Resumed from epoch {start_epoch}")

    for epoch in range(start_epoch, EPOCHS):
        student.train(); proj_head.train()
        run_loss = 0.0
        sw  = math.exp(-DECAY_RATE * epoch)
        trw = TOKEN_REL_WEIGHT * math.exp(-DECAY_RATE * epoch)  # also anneals

        for inputs, targets in trainloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            opt.zero_grad()
            with torch.no_grad(): t_out = teacher(inputs)
            out = student(inputs)
            s_cls, s_dist = out if isinstance(out, tuple) else (out, out)

            loss = (TASK_WEIGHT    * criterion_ce(s_cls, targets)
                  + DISTILL_WEIGHT * soft_kd_loss(s_dist, t_out)
                  + sw             * cosine_sem_loss(cache['t_pool'], cache['s_dist'], proj_head)
                  + trw            * token_relation_loss(cache['t_spatial'], cache['s_patches']))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            update_ema(student, ema_params,EMA_DECAY)
            run_loss += loss.item()

        sched.step()

        val_acc_live = 0.0
        student.eval()
        correct = total = 0
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                out = student(inputs)
                if isinstance(out, tuple): out = (out[0] + out[1]) / 2
                correct += out.max(1)[1].eq(targets).sum().item(); total += targets.size(0)
        val_acc_live = 100. * correct / total
        
        val_acc = validate_ema(student, ema_params, testloader)
        best_acc = max(best_acc, max(val_acc, val_acc_live))
        print(f"  Ep {epoch+1:2d} | Loss {run_loss/len(trainloader):.4f} | "
              f"Live {val_acc_live:.2f}% | EMA {val_acc:.2f}% | "
              f"SW {sw:.3f} | TRW {trw:.3f} | LR {opt.param_groups[0]['lr']:.5f}")
        torch.save({'epoch': epoch, 'model': student.state_dict(), 'proj': proj_head.state_dict(),
                    'opt': opt.state_dict(), 'sched': sched.state_dict(),
                    'ema': ema_params, 'best_acc': best_acc}, CKPT)

    results_p3[DATA_SCALE] = best_acc
    print(f"  ✓ Best: {best_acc:.2f}%")

print(f"\n{'='*65}\nP3 SUMMARY\n{'='*65}")
for s, a in results_p3.items(): print(f"  {s:>5}: {a:.2f}%")


  P3 | 10%
  5000 images | 79 batches/epoch
  EMA_DECAY set to 0.995571 for 790 total updates
  Ep  1 | Loss 8.3864 | Live 4.05% | EMA 3.75% | SW 1.000 | TRW 0.300 | LR 0.00078
  Ep  2 | Loss 7.5266 | Live 4.79% | EMA 4.40% | SW 0.905 | TRW 0.271 | LR 0.00071
  Ep  3 | Loss 7.1328 | Live 6.64% | EMA 5.12% | SW 0.819 | TRW 0.246 | LR 0.00060
  Ep  4 | Loss 6.7940 | Live 7.78% | EMA 6.07% | SW 0.741 | TRW 0.222 | LR 0.00047
  Ep  5 | Loss 6.5140 | Live 9.71% | EMA 7.27% | SW 0.670 | TRW 0.201 | LR 0.00033
  Ep  6 | Loss 6.2660 | Live 9.89% | EMA 8.07% | SW 0.607 | TRW 0.182 | LR 0.00020
  Ep  7 | Loss 6.0318 | Live 10.66% | EMA 9.10% | SW 0.549 | TRW 0.165 | LR 0.00009
  Ep  8 | Loss 5.8731 | Live 11.54% | EMA 9.91% | SW 0.497 | TRW 0.149 | LR 0.00003
  Ep  9 | Loss 5.7543 | Live 11.69% | EMA 10.12% | SW 0.449 | TRW 0.135 | LR 0.00000
  Ep 10 | Loss 5.7310 | Live 11.74% | EMA 10.65% | SW 0.407 | TRW 0.122 | LR 0.00003
  ✓ Best: 11.74%

  P3 | 25%
  12500 images | 196 batches/epoch
  EMA